In [7]:
%pip install pypdf langchain-text-splitters

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 461 kB 9.6 MB/s eta 0:00:01
     |████████████████████████████████| 66 kB 18.8 MB/s eta 0:00:01
     |████████████████████████████████| 396 kB 106.9 MB/s eta 0:00:01
     |████████████████████████████████| 572 kB 30.8 MB/s eta 0:00:01
     |████████████████████████████████| 472 kB 34.6 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 104.4 MB/s eta 0:00:01
     |████████████████████████████████| 54 kB 15.5 MB/s eta 0:00:01
     |████████████████████████████████| 640 kB 102.4 MB/s eta 0:00:01
     |████████████████████████████████| 64 kB 10.8 MB/s eta 0:00:01
     |████████████████████████████████| 245 kB 59.2 MB/s eta 0:00:01
     |████████████████████████████████| 73 kB 11.6 MB/s eta 0:00:01
     |████████████████████████████████| 136 kB 122.4 MB/s eta 0:00:01
     |████████████████████████████████| 113 kB 37.8 MB/s eta 0:00:01
     |████████████████████

In [13]:
%pip install sentence-transformers chromadb


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 21.7 MB 3.3 MB/s eta 0:00:01
     |████████████████████████████████| 90 kB 1.2 MB/s eta 0:00:011
     |████████████████████████████████| 60 kB 16.1 MB/s eta 0:00:01
  Using cached typer-0.23.2-py3-none-any.whl (56 kB)
     |████████████████████████████████| 40 kB 5.5 MB/s  eta 0:00:01
     |████████████████████████████████| 12.0 MB 29.7 MB/s eta 0:00:01
     |████████████████████████████████| 68 kB 13.3 MB/s eta 0:00:01
     |████████████████████████████████| 40 kB 14.1 MB/s eta 0:00:01
     |████████████████████████████████| 48 kB 39.6 MB/s eta 0:00:01
     |████████████████████████████████| 495 kB 30.6 MB/s eta 0:00:01
     |████████████████████████████████| 16.8 MB 29.9 MB/s eta 0:00:01
     |████████████████████████████████| 180 kB 31.8 MB/s eta 0:00:01
     |████████████████████████████████| 69 kB 27.6 MB/s eta 0:00:01
     |████████████████████████████████| 2.0 MB

In [3]:
import os
os.listdir()

['.DS_Store', 'literature', 'RAG_pipiline.ipynb']

In [ ]:
from pypdf import PdfReader
from transformers import AutoTokenizer
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf_path = "literature/metformin literature.pdf"
reader = PdfReader(pdf_path)

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2"
)

splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer,
    chunk_size=200,       # Maximum tokens per chunk
    chunk_overlap=40,     # Token overlap between adjacent chunks
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = []

for page_number, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ""

    if not text.strip():
        continue

    documents = splitter.create_documents(
        texts=[text],
        metadatas=[{
            "source": pdf_path,
            "page": page_number,
        }],
    )

    chunks.extend(documents)

print(f"Created {len(chunks)} chunks")

if chunks:
    print(chunks[0].page_content)
    print(chunks[0].metadata)

Created 239 chunks
Citation: Papadakos, S.P .; Argyrou,
A.; Lekakis, V .; Arvanitakis, K.;
Kalisperati, P .; Stergiou, I.E.;
Konstantinidis, I.; Schizas, D.;
Koufakis, T.; Germanidis, G.; et al.
Metformin in Esophageal Carcinoma:
Exploring Molecular Mechanisms and
Therapeutic Insights. Int. J. Mol. Sci.
2024, 25, 2978. https://doi.org/
10.3390/ijms25052978
Academic Editor: Hidekazu Suzuki
Received: 10 February 2024
Revised: 29 February 2024
Accepted: 3 March 2024
Published: 4 March 2024
Copyright: © 2024 by the authors.
Licensee MDPI, Basel, Switzerland.
This article is an open access article
{'source': 'literature/metformin literature.pdf', 'page': 1}


In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

texts = [chunk.page_content for chunk in chunks]

embeddings = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
)

print(embeddings.shape)  # (number_of_chunks, 384)

Batches: 100%|██████████| 8/8 [00:01<00:00,  6.61it/s]

(239, 384)


In [14]:
import chromadb

# Save the database in a local folder
client = chromadb.PersistentClient(path="./chroma_db")

# We supply our own embeddings from all-MiniLM-L6-v2
collection = client.get_or_create_collection(
    name="metformin_minilm",
    embedding_function=None,
)

assert len(chunks) == len(embeddings), "Each chunk needs one embedding."

# Store in batches to avoid oversized requests
batch_size = 100

for start in range(0, len(chunks), batch_size):
    end = min(start + batch_size, len(chunks))

    collection.upsert(
        ids=[f"metformin_chunk_{i}" for i in range(start, end)],
        embeddings=embeddings[start:end].tolist(),
        documents=[chunk.page_content for chunk in chunks[start:end]],
        metadatas=[chunk.metadata for chunk in chunks[start:end]],
    )

print(f"Stored {collection.count()} chunks in ./chroma_db")

Stored 239 chunks in ./chroma_db
